# Step 1. Embedding: 뜻이 비슷한 글 찾기
강의에서 본 "글을 좌표로 바꾸면 비슷한 글이 가까이 모인다"를 직접 확인합니다.

## 핵심 개념
- **Embedding(임베딩)**: 글을 숫자 묶음(벡터)으로 바꾸는 것. 뜻이 비슷하면 숫자도 비슷해지도록 학습된 모델을 씁니다.
- **Cosine(코사인) 유사도**: 두 벡터가 가리키는 방향이 얼마나 같은지. 1에 가까울수록 비슷합니다.
- **키워드 검색과의 차이**: "숙제"로 검색하면 "과제"가 적힌 글은 못 찾습니다. Embedding은 찾습니다.

In [ ]:
# 필요한 패키지를 설치합니다. 처음에는 1분쯤 걸립니다.
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "ollama", "numpy", "matplotlib"], check=True)
print("✅ 패키지 준비 완료. 다음 셀로 넘어가세요.")

## 1. 설정

In [ ]:
import json

try:
    with open("../config.json") as f:
        _config = json.load(f)
    print("✅ config.json 로드 완료")
except FileNotFoundError:
    raise SystemExit("❌ config.json이 없습니다. Step 0 노트북을 먼저 실행해 주세요.")

EMBED_MODEL = _config["EMBED_MODEL"]
LLM_MODEL = _config["LLM_MODEL"]
AGENT_MODEL = _config["AGENT_MODEL"]
print(f"   Embedding 모델: {EMBED_MODEL}")
print(f"   LLM 모델:       {LLM_MODEL}")
print(f"   Agent 모델:     {AGENT_MODEL}")

## 2. 문장을 벡터로 바꾸기
`embed()` 함수 하나면 됩니다. 뒤의 Step에서도 계속 씁니다.

In [ ]:
import numpy as np
import ollama


def embed(texts):
    """문장 목록을 벡터 목록으로 바꾼다."""
    return np.array(ollama.embed(model=EMBED_MODEL, input=texts)["embeddings"])


def cosine(a, b):
    """두 벡터의 Cosine 유사도 (1에 가까울수록 비슷)."""
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))


# TODO: 문장을 바꾸거나 추가해 보세요!
memos = [
    "다음 주 화요일까지 RAG 과제 제출",
    "동아리 회식 장소는 학교 앞 삼겹살집",
    "병원 주차장은 지하 2층",
    "구내식당 점심은 11시 30분부터",
    "흉부 엑스레이에서 폐 아래쪽이 하얗게 보이면 폐렴을 의심",
    "혈압약은 아침 식사 후에 복용",
]
memo_vecs = embed(memos)
print(f"✅ 문장 {len(memos)}개를 {memo_vecs.shape[1]}차원 벡터로 바꿨습니다.")

## 3. 질문과 가장 가까운 문장 찾기
질문에 "과제"라는 단어는 없습니다. 그래도 찾는지 보세요.

In [ ]:
def search(query, top=3):
    q = embed([query])[0]
    scores = [cosine(q, v) for v in memo_vecs]
    order = np.argsort(scores)[::-1][:top]
    print(f"🔎 질문: {query}")
    for rank, i in enumerate(order, 1):
        print(f"   {rank}위  유사도 {scores[i]:.3f}  {memos[i]}")
    print()


# TODO: 질문을 바꿔 보세요!
search("숙제 마감이 언제지?")
search("차는 어디에 대요?")
search("pneumonia가 의심되는 사진의 특징")

## 4. 키워드 검색이었다면?
같은 질문을 "겹치는 단어 수"로 찾아봅니다. 겹치는 단어가 없으면 0점입니다.

In [ ]:
def keyword_search(query):
    q_words = set(query.replace("?", "").split())
    print(f"🔎 질문: {query}")
    for m in memos:
        overlap = q_words & set(m.split())
        print(f"   겹치는 단어 {len(overlap)}개  {m}")
    print()


keyword_search("숙제 마감이 언제지?")
print("→ 겹치는 단어가 하나도 없어서 키워드 검색으로는 못 찾습니다.")
print("  반대로 약 이름, 질병 코드처럼 글자가 정확히 맞아야 하면 키워드 검색이 더 낫습니다.")

## 5. 지도로 그려 보기
1,024차원을 2차원으로 눌러서 그립니다. 비슷한 주제끼리 가까이 모이는지 보세요.

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "AppleGothic"   # 맥의 한글 글꼴
plt.rcParams["axes.unicode_minus"] = False

query = "숙제 마감이 언제지?"
all_texts = memos + [query]
X = embed(all_texts)
X = X - X.mean(axis=0)
U, S, _ = np.linalg.svd(X, full_matrices=False)   # PCA: 가장 잘 퍼지는 두 방향만 남김
xy = U[:, :2] * S[:2]

plt.figure(figsize=(8, 5))
plt.scatter(xy[:-1, 0], xy[:-1, 1], s=60, color="#1F4ED8")
plt.scatter(xy[-1, 0], xy[-1, 1], s=120, color="#0E8A74", marker="*")
for (x, y), t in zip(xy, all_texts):
    plt.annotate(t, (x, y), xytext=(6, 6), textcoords="offset points", fontsize=9)
plt.title("문장의 의미 지도 (2차원으로 줄인 그림)")
plt.xticks([]); plt.yticks([])
plt.show()

## 정리
- 글을 벡터로 바꾸면 "비슷한 것 찾기"는 "가까운 점 찾기"가 됩니다.
- 문장이 수백만 개가 되면 하나하나 비교할 수 없습니다. 그래서 **Vector DB**가 필요합니다. → Step 2